# Notebook 05 - How Generative AI Works

Goes with **Lesson 05**. No API key needed. We build tiny versions of the ideas inside real LLMs.

In [ ]:
from pathlib import Path

# The data folder is next to the notebooks folder. In Colab, upload the files to the current folder.
DATA_DIR = Path("../data") if Path("../data").exists() else Path(".")
DOCS_DIR = DATA_DIR / "documents" if (DATA_DIR / "documents").exists() else DATA_DIR
print("Data folder:", DATA_DIR.resolve())

## 1. A tiny generative language model

Real LLMs predict the next **token** using a huge Transformer. Our tiny model predicts the next **word** using simple counts: "after the word X, which words came next, and how often?"

In [ ]:
import re
import random
from collections import defaultdict, Counter

text = " ".join(f.read_text(encoding="utf-8") for f in sorted(DOCS_DIR.glob("*.txt")))
words = re.findall(r"[a-zA-Z']+|[.]", text.lower())
print("Training words:", len(words))

next_words = defaultdict(Counter)
for current, following in zip(words, words[1:]):
    next_words[current][following] += 1

print()
print("After 'the', the most common next words are:", next_words["the"].most_common(6))
print("After 'library', the next words are:", next_words["library"].most_common(5))

Now we **generate**: start with a word, pick the next word using the probabilities, add it, and repeat. This is exactly the LLM loop, just much simpler.

**Temperature** controls how adventurous the choice is.

In [ ]:
import numpy as np

def pick_next(word, temperature):
    options = next_words[word]
    if not options:
        return None
    candidates = list(options.keys())
    counts = np.array(list(options.values()), dtype=float)
    # temperature: low = sharpen the probabilities, high = flatten them
    weights = counts ** (1.0 / temperature)
    probabilities = weights / weights.sum()
    return random.choices(candidates, weights=probabilities)[0]

def generate(start, max_words=20, temperature=1.0):
    output = [start]
    for _ in range(max_words):
        nxt = pick_next(output[-1], temperature)
        if nxt is None:
            break
        output.append(nxt)
        if nxt == ".":
            break
    return " ".join(output).replace(" .", ".")

random.seed(4)
print("Low temperature (0.2) - safe and repetitive:")
for _ in range(3):
    print("  ", generate("the", temperature=0.2))
print()
print("High temperature (2.0) - varied and surprising:")
for _ in range(3):
    print("  ", generate("the", temperature=2.0))

The sentences are **new** (not copied), and they sound like the training text. But they can also be **nonsense** or **false** - the model only knows which words usually follow each other. This is a small picture of why big LLMs can **hallucinate**.

## 2. Temperature on next-token probabilities

Imagine the model's raw scores for the next token after "The capital of India is".

In [ ]:
import matplotlib.pyplot as plt

tokens = [" New", " Delhi", " a", " the", " Mumbai"]
scores = np.array([6.0, 3.0, 2.0, 1.5, 1.0])     # raw model scores (called logits)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, temperature in zip(axes, [0.3, 1.0, 2.5]):
    p = np.exp(scores / temperature)
    p = p / p.sum()
    ax.bar(tokens, p)
    ax.set_ylim(0, 1)
    ax.set_title(f"temperature = {temperature}")
plt.suptitle("Low temperature makes the top token almost certain; high temperature spreads the chances")
plt.show()

## 3. Tokens

LLMs split text into **tokens**. A common rule: 1 token is about 3/4 of an English word. If you have an API key, the cell below also asks Gemini for the **exact** token count.

In [ ]:
sample = "Tokenisation splits unbelievably long words into smaller pieces called tokens."
word_count = len(sample.split())
print("Words:", word_count)
print("Estimated tokens (words x 4/3):", round(word_count * 4 / 3))

import ai_helper
if ai_helper.API_KEY:
    try:
        result = ai_helper._get_client().models.count_tokens(model=ai_helper.MODEL, contents=sample)
        print("Exact tokens from Gemini:", result.total_tokens)
    except Exception as error:
        print("Could not count tokens:", type(error).__name__)
else:
    print("[MOCK MODE] Add an API key (Lesson 06) to see the exact count.")

## 4. Attention: which words matter for "it"?

In a Transformer, each word makes a **query** ("what am I looking for?") and each word has a **key** ("what do I contain?"). Attention = softmax of (query . key). A real model **learns** these vectors. Here we make tiny 2-number vectors by hand: `[is a living thing, is a place]`.

In [ ]:
import pandas as pd

sentence = ["the", "animal", "did", "not", "cross", "the", "street", "because", "it", "was", "tired"]
keys = {
    "animal": [1.0, 0.0],     # a living thing
    "street": [0.0, 1.0],     # a place
}
key_vectors = np.array([keys.get(w, [0.1, 0.1]) for w in sentence])

def attention(query):
    scores = key_vectors @ np.array(query) / np.sqrt(2)
    weights = np.exp(scores) / np.exp(scores).sum()
    return pd.Series(weights, index=sentence).round(2)

# "tired" describes living things, so the query of "it" looks for a living thing
tired = attention([3.0, 0.0])
# "wide" describes places, so the query of "it" looks for a place
wide = attention([0.0, 3.0])

table = pd.DataFrame({"...because it was TIRED": tired.values, "...because it was WIDE": wide.values}, index=sentence)
table

In [ ]:
print("With 'tired', 'it' pays most attention to:", tired.idxmax())
print("With 'wide',  'it' pays most attention to:", wide.idxmax())

Real Transformers do this for **every** word, with vectors of hundreds of numbers, in many layers and many "attention heads" at the same time.

## Summary

| Real LLM | Our tiny version |
|---|---|
| Predicts the next token | Predicts the next word |
| Probabilities from a Transformer with billions of parameters | Probabilities from word counts |
| Temperature controls randomness | Same |
| Attention decides which earlier tokens matter | Hand-made attention example |

Go back to **Lesson 05** and do the **Check** section.